In [1]:
%matplotlib inline

import sys
import logging
import itertools
import copy

import numpy as np
np.random.seed(0)
import pandas as pd
import gymnasium as gym
import matplotlib.pyplot as plt
import torch
torch.manual_seed(0)
import torch.nn as nn
import torch.optim as optim

logging.basicConfig(level=logging.INFO,
        format='%(asctime)s [%(levelname)s] %(message)s',
        stream=sys.stdout, datefmt='%H:%M:%S')

In [2]:
env = gym.make('Pendulum-v1',render_mode="human")
for key in vars(env.spec):
    logging.info('%s: %s', key, vars(env.spec)[key])
for key in vars(env.unwrapped):
    logging.info('%s: %s', key, vars(env.unwrapped)[key])

14:54:49 [INFO] id: Pendulum-v1
14:54:49 [INFO] entry_point: gymnasium.envs.classic_control.pendulum:PendulumEnv
14:54:49 [INFO] reward_threshold: None
14:54:49 [INFO] nondeterministic: False
14:54:49 [INFO] max_episode_steps: 200
14:54:49 [INFO] order_enforce: True
14:54:49 [INFO] disable_env_checker: False
14:54:49 [INFO] kwargs: {'render_mode': 'human'}
14:54:49 [INFO] additional_wrappers: ()
14:54:49 [INFO] vector_entry_point: None
14:54:49 [INFO] namespace: None
14:54:49 [INFO] name: Pendulum
14:54:49 [INFO] version: 1
14:54:49 [INFO] max_speed: 8
14:54:49 [INFO] max_torque: 2.0
14:54:49 [INFO] dt: 0.05
14:54:49 [INFO] g: 10.0
14:54:49 [INFO] m: 1.0
14:54:49 [INFO] l: 1.0
14:54:49 [INFO] render_mode: human
14:54:49 [INFO] screen_dim: 500
14:54:49 [INFO] screen: None
14:54:49 [INFO] clock: None
14:54:49 [INFO] isopen: True
14:54:49 [INFO] action_space: Box(-2.0, 2.0, (1,), float32)
14:54:49 [INFO] observation_space: Box([-1. -1. -8.], [1. 1. 8.], (3,), float32)
14:54:49 [INFO] spec

In [3]:
#经验回放池
class DQNReplayer:
    def __init__(self, capacity):
        #用DataFrame存储经验，索引范围是[0, capacity-1]，列对应经验的5个核心字段
        self.memory = pd.DataFrame(index=range(capacity),
                columns=['observation', 'action', 'reward',
                'next_observation', 'terminated'])
        #记录当前要写入的位置（指针）
        self.i = 0
        #记录已存储的经验总数（不会超过capacity）
        self.count = 0
        #回放池的最大容量
        self.capacity = capacity
    #存储单条经验
    def store(self, *args):
        #将传入的经验参数（obs, act, rew, next_obs, terminated）转为数组，存入当前指针位置
        self.memory.loc[self.i] = np.asarray(args, dtype=object)
        #指针后移一位，超过容量则循环到0
        self.i = (self.i + 1) % self.capacity
        #经验数+1，最多不超过容量
        self.count = min(self.count + 1, self.capacity)
    #随机采样批量经验
    def sample(self, size):
        #从已存储的count条经验中，随机选size个索引（无放回采样）
        indices = np.random.choice(self.count, size=size)
        #按列提取采样的经验，转为数组并返回（返回5个数组，对应5个字段）
        return (np.stack(self.memory.loc[indices, field]) for field in
                self.memory.columns)

In [4]:
#OU噪声
class OrnsteinUhlenbeckProcess:
    def __init__(self, x0):
        self.x = x0 #噪声的初始值（通常是全0数组，维度和动作维度一致）
    #mu噪声的均值 sigma波动率 theta回归系数 dt时间步长
    def __call__(self, mu=0., sigma=1., theta=.15, dt=.01):
        # 生成标准正态分布的随机数（维度和self.x一致）
        n = np.random.normal(size=self.x.shape)
        # OU过程的核心更新公式
        self.x += (theta * (mu - self.x) * dt + sigma * np.sqrt(dt) * n)
        #返回更新后的噪声值
        return self.x

In [5]:
class DDPGAgent:
    def __init__(self, env):
        state_dim = env.observation_space.shape[0] #状态维度
        self.action_dim = env.action_space.shape[0] #动作维度
        self.action_low = env.action_space.low[0] #动作最小值
        self.action_high = env.action_space.high[0] #动作最大值
        self.gamma = 0.99 #折扣因子

        self.replayer = DQNReplayer(20000) #经验回放池
#Actor网络（策略网络）
        self.actor_evaluate_net = self.build_net(
                input_size=state_dim, hidden_sizes=[32, 64],
                output_size=self.action_dim)
        self.actor_optimizer = optim.Adam(self.actor_evaluate_net.parameters(),
                lr=0.0001) #优化器
        self.actor_target_net = copy.deepcopy(self.actor_evaluate_net) #目标网
#Critic网络（价值网络） 输入状态+动作 输出动作的价值
        self.critic_evaluate_net = self.build_net(
                input_size=state_dim+self.action_dim, hidden_sizes=[64, 128])
        self.critic_optimizer = optim.Adam(self.critic_evaluate_net.parameters(),
                lr=0.001)
        self.critic_loss = nn.MSELoss() #损失函数：均方误差
        self.critic_target_net = copy.deepcopy(self.critic_evaluate_net) #目标网

    def build_net(self, input_size, hidden_sizes, output_size=1, 
            output_activator=None): #构建全连接神经网络
        layers = []
        #依次构建线性层+ReLU激活(隐藏层)
        for input_size, output_size in zip(
                [input_size,] + hidden_sizes, hidden_sizes + [output_size,]):
            layers.append(nn.Linear(input_size, output_size)) #线性层（特征映射）
            layers.append(nn.ReLU()) #非线性激活（增加表达能力）
        layers = layers[:-1] #去掉最后一个ReLU（避免输出被激活，比如动作/价值不需要ReLU截断）
        if output_activator:
            layers.append(output_activator) #可选输出激活
        net = nn.Sequential(*layers) #组装网络
        return net

    def reset(self, mode=None): #重置Agent状态（训练/测试模式切换）
        self.mode = mode
        if self.mode == 'train':
            self.trajectory = [] #存储轨迹
            #OU噪声：连续动作空间的探索噪声
            self.noise = OrnsteinUhlenbeckProcess(np.zeros((self.action_dim,)))
#核心交互逻辑（输入环境观测，输出动作）
    def step(self, observation, reward, terminated):
        #探索阶段：经验池样本不足时，纯随机动作（先收集足够经验）
        if self.mode == 'train' and self.replayer.count < 3000:
            action = np.random.uniform(self.action_low, self.action_high)
        #正常决策：用Actor评估网输出确定性动作
        else:
            state_tensor = torch.as_tensor(observation,
                    dtype=torch.float).reshape(1, -1)
            action_tensor = self.actor_evaluate_net(state_tensor)
            action = action_tensor.detach().numpy()[0]
        #训练模式下的额外操作：加噪声、存经验、触发学习
        if self.mode == 'train':
            # noisy action
            noise = self.noise(sigma=0.1) #加探索噪声
            action = (action + noise).clip(self.action_low, self.action_high) #裁剪到动作范围

            self.trajectory += [observation, reward, terminated, action] #存储轨迹
            #每收集2步经验（8个元素），存回回放池
            if len(self.trajectory) >= 8:
                state, _, _, act, next_state, reward, terminated, _ = \
                        self.trajectory[-8:]
                self.replayer.store(state, act, reward, next_state, terminated)
            #经验池足够时，触发学习
            if self.replayer.count >= 3000:
                self.learn()
        return action

    def close(self):
        pass
#软更新目标网络
    def update_net(self, target_net, evaluate_net, learning_rate=0.005):
        for target_param, evaluate_param in zip(
                target_net.parameters(), evaluate_net.parameters()):
            #目标网参数 = 学习率 * 评估网参数 + （1 - 学习率）* 目标网参数
            target_param.data.copy_(learning_rate * evaluate_param.data
                    + (1 - learning_rate) * target_param.data)
#DDPG核心，分两步更新Critic 和 Actor 
    def learn(self):
        # replay 从回放池采集批量经验（64条）
        states, actions, rewards, next_states, terminateds = \
                self.replayer.sample(64)
        #转tensor
        state_tensor = torch.as_tensor(states, dtype=torch.float)
        action_tensor = torch.as_tensor(actions, dtype=torch.float)
        reward_tensor = torch.as_tensor(rewards, dtype=torch.float)
        next_state_tensor = torch.as_tensor(next_states, dtype=torch.float)
        terminated_tensor = torch.as_tensor(terminateds, dtype=torch.float)

        # update critic 评估动作价值
        #用Actor目标网预测下一状态的动作，加少量噪声（探索）
        next_action_tensor = self.actor_target_net(next_state_tensor)
        noise_tensor = (0.2 * torch.randn_like(action_tensor, dtype=torch.float))
        noisy_next_action_tensor = (next_action_tensor + noise_tensor).clamp(
                self.action_low, self.action_high)
        #拼接下一状态+下一动作，用Critic目标网计算Q值
        next_state_action_tensor = torch.cat([next_state_tensor,
                noisy_next_action_tensor], 1)
        next_q_tensor = self.critic_target_net(next_state_action_tensor).squeeze(1)
        #贝尔曼方程： 目标Q值 = 即时奖励 + 折扣 * 未来Q值
        critic_target_tensor = reward_tensor + (1. - terminated_tensor) * \
                self.gamma * next_q_tensor
        critic_target_tensor = critic_target_tensor.detach() #固定目标网梯度
        #计算Critic预测Q值，求损失并更新
        state_action_tensor = torch.cat([state_tensor, action_tensor], 1)
        critic_pred_tensor = self.critic_evaluate_net(state_action_tensor
                ).squeeze(1)
        critic_loss_tensor = self.critic_loss(critic_pred_tensor,
                critic_target_tensor)
        self.critic_optimizer.zero_grad() #清空梯度
        critic_loss_tensor.backward() #反向传播
        self.critic_optimizer.step() #更新参数

        # update actor 优化策略，输出更优动作
        pred_action_tensor = self.actor_evaluate_net(state_tensor)
        pred_action_tensor = pred_action_tensor.clamp(self.action_low,
                self.action_high)
        #拼接 当前状态+预测动作，用Critic评估该动作的价值
        pred_state_action_tensor = torch.cat([state_tensor, pred_action_tensor], 1)
        critic_pred_tensor = self.critic_evaluate_net(pred_state_action_tensor)
        #Actor损失，负Q值均值（因为要最大化Q值，所以最小化-Q值）
        actor_loss_tensor = -critic_pred_tensor.mean()
        self.actor_optimizer.zero_grad()
        actor_loss_tensor.backward()
        self.actor_optimizer.step()
        #软更新目标网络
        self.update_net(self.critic_target_net, self.critic_evaluate_net)
        self.update_net(self.actor_target_net, self.actor_evaluate_net)


agent = DDPGAgent(env)

In [6]:
def play_episode(env, agent, seed=None, mode=None, render=False):
    #重置环境：获取初始观测（seed保证可复现）
    observation, _ = env.reset(seed=seed)
    #初始化奖励、终止标志（terminated=任务完成/失败，truncated=步数超限）
    reward, terminated, truncated = 0., False, False
    #重置Agent：传入训练/测试模式（比如训练模式加噪声，测试模式不加）
    agent.reset(mode=mode)
    #初始化该局的总奖励和步数
    episode_reward, elapsed_steps = 0., 0
    #单局交互循环（直到终止）
    while True:
        #Agent根据当前观测、上一步奖励、终止标志，输出动作
        action = agent.step(observation, reward, terminated)
        #可选渲染环境（比如显示游戏画面、机器人运动）
        if render:
            env.render()
        #若环境已终止，跳出循环
        if terminated or truncated:
            break
        #环境执行动作，返回新的观测、奖励、终止标志
        observation, reward, terminated, truncated, _ = env.step(action)
        #. 累计该局奖励和步数
        episode_reward += reward
        elapsed_steps += 1
    #关闭Agent（预留接口，比如释放资源）
    agent.close()
    #返回该局总奖励和步数
    return episode_reward, elapsed_steps


logging.info('==== train ====')
episode_rewards = [] ## 存储每局的奖励，用于监控训练进度
## itertools.count()：无限循环（直到满足终止条件）
for episode in itertools.count():
    # 运行一局训练（mode='train'，Agent会加噪声、收集经验、更新网络）
    episode_reward, elapsed_steps = play_episode(env, agent, seed=episode,
            mode='train')
    episode_rewards.append(episode_reward)
    # 打印训练日志（局数、奖励、步数)
    logging.info('train episode %d: reward = %.2f, steps = %d',
            episode, episode_reward, elapsed_steps)
    # 打印训练日志（局数、奖励、步数
    if np.mean(episode_rewards[-10:]) > -120:
        break
# 绘制奖励变化曲线（直观查看训练趋势)
plt.plot(episode_rewards)


logging.info('==== test ====')
episode_rewards = []
for episode in range(100):
    episode_reward, elapsed_steps = play_episode(env, agent)
    episode_rewards.append(episode_reward)
    logging.info('test episode %d: reward = %.2f, steps = %d',
            episode, episode_reward, elapsed_steps)
# 输出统计结果：平均奖励 ± 标准差（衡量性能和稳定性）
logging.info('average episode reward = %.2f ± %.2f',
        np.mean(episode_rewards), np.std(episode_rewards))

14:54:50 [INFO] ==== train ====
14:54:57 [INFO] train episode 0: reward = -978.58, steps = 200
14:55:04 [INFO] train episode 1: reward = -978.48, steps = 200
14:55:11 [INFO] train episode 2: reward = -1120.99, steps = 200
14:55:18 [INFO] train episode 3: reward = -1463.17, steps = 200
14:55:24 [INFO] train episode 4: reward = -1641.70, steps = 200
14:55:31 [INFO] train episode 5: reward = -1457.94, steps = 200
14:55:38 [INFO] train episode 6: reward = -860.93, steps = 200
14:55:45 [INFO] train episode 7: reward = -1145.15, steps = 200
14:55:51 [INFO] train episode 8: reward = -922.51, steps = 200
14:55:58 [INFO] train episode 9: reward = -1604.65, steps = 200
14:56:05 [INFO] train episode 10: reward = -1675.24, steps = 200
14:56:12 [INFO] train episode 11: reward = -1514.80, steps = 200
14:56:19 [INFO] train episode 12: reward = -1297.64, steps = 200
14:56:26 [INFO] train episode 13: reward = -1400.94, steps = 200
14:56:32 [INFO] train episode 14: reward = -1410.17, steps = 200
14:56:3

KeyboardInterrupt: 

In [9]:
env.close()